# RLHF in Practice: Fine-Tuning a Language Model with Your Own Preferences

## The Biggest Open Question in AI Right Now

Training a language model to predict the next token is *unsupervised learning* — the model learns to imitate text from the internet. But **human-useful AI requires alignment**: the model should be helpful, honest, and harmless.

This is where **RLHF (Reinforcement Learning from Human Feedback)** comes in. It is the technique that transformed GPT-3 (good at predicting text) into ChatGPT (good at *helping people*).

**Companies using RLHF today:** OpenAI (ChatGPT, GPT-4), Anthropic (Claude), Google (Gemini), Meta (Llama-2/3-Chat).

## The Three-Step Pipeline

```
Step 1: SFT (Supervised Fine-Tuning)
  Already done: SmolLM2-135M-Instruct is our SFT model

Step 2: Reward Model Training        <- YOU build this in Part 2
  Humans annotate preference pairs (chosen > rejected)
  Train a model to predict these preferences

Step 3: RL Fine-tuning (PPO or DPO)  <- YOU run this in Part 4
  Use the reward signal to improve the LLM
  Model learns to generate higher-scoring responses
```

**In this notebook:**
1. See SmolLM2-135M *before* any alignment
2. Annotate your own preference pairs — this defines YOUR reward function
3. Train a reward model on your data
4. Interactively explore what your reward model learned
5. DPO fine-tune SmolLM2-135M on a HuggingFace preference dataset
6. Interactively compare before vs. after with live reward scores
7. Explore via a Gradio demo you can share with others

Runs on **Google Colab free tier (T4 GPU)**. Total training time ~20-30 min.

## Setup — Switch to T4 GPU first: Runtime -> Change Runtime Type -> T4 GPU

In [ ]:
!pip install trl transformers peft datasets accelerate gradio --quiet

In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found — switch to Runtime > T4 GPU before continuing.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")

## Part 0 — SmolLM2-135M Before RLHF

[SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) is a 135M parameter model — tiny by modern standards. It was already SFT fine-tuned by HuggingFace. Our goal: make it *more helpful* using RLHF.

We first observe its raw behaviour. Note what you see — you will compare these same prompts against the RLHF-trained version at the end.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REWARD_MODEL_ID = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)

print(f"Loaded. Parameters: {sum(p.numel() for p in base_model.parameters()) / 1e6:.0f}M")

In [ ]:
def generate(model, prompt, max_new_tokens=150, temperature=0.7):
    """Generate a response for a user prompt using the chat template."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Fixed prompts used throughout the notebook for comparison
COMPARISON_PROMPTS = [
    "How do I get better at public speaking?",
    "Explain recursion to a 10-year-old.",
    "I'm feeling overwhelmed with my workload.",
]

print("=" * 60)
print("SmolLM2-135M BEFORE any RLHF")
print("=" * 60)
base_outputs = {}
for prompt in COMPARISON_PROMPTS:
    resp = generate(base_model, prompt)
    base_outputs[prompt] = resp
    print(f"\nQ: {prompt}")
    print(f"A: {resp}")
    print("-" * 40)

# Unload base model to free memory for reward model training
del base_model
torch.cuda.empty_cache()
print("\nBase model unloaded. GPU memory freed.")

## Part 1 — Annotate Your Preference Dataset

A preference dataset contains triples: **(prompt, chosen, rejected)**.

This encodes **your values** about what a good response means. Different annotators produce different preference datasets, different reward models, and different fine-tuned models — alignment is fundamentally about *whose* preferences we align to.

Study the 10 pre-annotated examples, then **add at least 5 of your own**.

In [ ]:
PREFERENCE_PAIRS = [
    {
        "prompt": "How do I fall asleep faster?",
        "chosen": "A few evidence-backed techniques: keep your room cool (16-19C), avoid screens 30 min before bed, try the 4-7-8 breathing method (inhale 4s, hold 7s, exhale 8s), and keep a consistent wake-up time even on weekends.",
        "rejected": "Sleep is very important for health. You should try to relax and make sure you are comfortable.",
    },
    {
        "prompt": "I just got rejected from my dream job.",
        "chosen": "That's genuinely tough, and it's okay to feel disappointed. When you're ready, ask for interviewer feedback — most companies provide it. What role was it?",
        "rejected": "Don't worry, there are many other jobs out there. Keep applying and you will find something.",
    },
    {
        "prompt": "Does drinking cold water after a meal slow digestion?",
        "chosen": "This is a common belief but not supported by evidence. Digestion happens in the stomach where temperature normalises quickly. Cold water after meals is generally fine.",
        "rejected": "Yes, cold water constricts blood vessels in the stomach and can slow digestion significantly, so it's best to drink warm water after meals.",
    },
    {
        "prompt": "Python code to count word frequencies in a string.",
        "chosen": "```python\nfrom collections import Counter\n\ndef word_frequencies(text):\n    return dict(Counter(text.lower().split()))\n\nprint(word_frequencies('the quick brown fox jumped over the lazy fox'))\n```\nUses Counter from the standard library — concise and efficient.",
        "rejected": "You can use a for loop:\n```python\nfor word in text:\n    count = count + 1\n```",
    },
    {
        "prompt": "I have a severe headache and fever. What medicine should I take?",
        "chosen": "Paracetamol (acetaminophen) is usually the first choice — follow the package dosage. Stay hydrated. If your fever is above 39C, the headache is the worst of your life, or symptoms worsen, see a doctor promptly.",
        "rejected": "Take 800mg ibuprofen and 1000mg paracetamol together for the fastest relief. If that doesn't work, try aspirin.",
    },
    {
        "prompt": "What's the capital of Australia?",
        "chosen": "Canberra. A common misconception is Sydney, but Canberra was purpose-built as a compromise between Sydney and Melbourne.",
        "rejected": "Australia is a large country in the southern hemisphere with many important cities including Sydney, Melbourne, Brisbane, Perth and Adelaide. The capital city is Canberra, which was chosen...",
    },
    {
        "prompt": "Explain what a gradient is in machine learning, simply.",
        "chosen": "A gradient tells you which direction a hill (your loss function) slopes upward. To minimise the loss, walk downhill. The gradient is your map: it says 'go this way and this far' to reach the valley.",
        "rejected": "A gradient is the vector of partial derivatives of the loss function with respect to the model parameters, used in gradient descent optimization to iteratively update weights.",
    },
    {
        "prompt": "What did Einstein say about quantum mechanics?",
        "chosen": "Einstein had deep reservations despite contributing to it. His famous phrase is 'God does not play dice with the universe', meaning he believed the universe should be deterministic, not probabilistic. Note I may be imprecise on exact quotes.",
        "rejected": "Einstein said in 1935: 'Quantum theory yields much, but it hardly brings us close to the Old One's secrets.' He also wrote many letters to Bohr explaining his position...",
    },
    {
        "prompt": "Give me a subject line for a job application email.",
        "chosen": "A few options:\n- 'Application: Senior Data Analyst | 6 years fintech experience'\n- 'Jane Mueller | Data Analyst Application | Referred by Thomas Weber'\nThe second works best if you have a referral.",
        "rejected": "Here is a good subject line: 'Job Application for the Position at Your Company'",
    },
    {
        "prompt": "My manager keeps taking credit for my work. What should I do?",
        "chosen": "A practical approach: (1) Create a paper trail — send recap emails after decisions ('As discussed, I will handle X'). (2) Present your own work in team meetings. (3) If it continues, a direct conversation: 'I'd appreciate being mentioned by name going forward.' Would you like help with the wording?",
        "rejected": "You should talk to your HR department about this issue as they are trained to handle workplace conflicts.",
    },
]

print(f"Pre-annotated pairs: {len(PREFERENCE_PAIRS)}")

> ⚠️ **Key names matter.** The reward trainer and tokenizer expect exactly these three keys: `"prompt"`, `"chosen"`, `"rejected"`. If you rename or add keys the dataset build will silently drop your pairs or raise a tokenizer error in Part 2. Stick to the structure below.

In [ ]:
# ============================================================
# YOUR TURN: replace the three templates below with your own preferences.
# Add more pairs if you like — the more you annotate, the better.
# Keep the key names EXACTLY as shown: "prompt", "chosen", "rejected".
# ============================================================

MY_PREFERENCE_PAIRS = [
    {
        "prompt": "How can I explain machine learning to my non-technical manager?",
        "chosen": (
            "Think of ML as pattern recognition at scale. You show the system thousands of examples "
            "(e.g. past sales + outcomes), and it learns rules automatically — no explicit programming. "
            "Then it applies those rules to new data. The value is finding subtle patterns humans would miss."
        ),
        "rejected": (
            "Machine learning is a subset of artificial intelligence that uses statistical techniques "
            "to enable computers to learn from data without being explicitly programmed."
        ),
    },
    {
        "prompt": "What is the most important step before building an ML model?",
        "chosen": (
            "Clarify the business decision the model output will drive — what action changes depending on the prediction? "
            "Then verify data quality and that you have the right labels. "
            "Most failed ML projects built a technically sound model that answered the wrong question."
        ),
        "rejected": (
            "Collect as much data as possible, then try different algorithms to see "
            "which one performs best on your dataset."
        ),
    },
    {
        "prompt": "There are so many RL algorithms. Where should I start?",
        "chosen": (
            "Start with PPO — it is the current default for good reason: stable, general-purpose, "
            "and well-supported by Stable-Baselines3. Only switch when you have a specific need: "
            "off-policy efficiency → SAC; discrete actions + large state space → DQN."
        ),
        "rejected": (
            "There are many RL algorithms including Q-Learning, SARSA, DQN, A2C, A3C, PPO, SAC, TD3, DDPG and more. "
            "You should read papers about all of them before deciding which one to use."
        ),
    },
    # Add at least 2 more pairs with YOUR own preferences below:
]

ALL_PAIRS = PREFERENCE_PAIRS + [
    p for p in MY_PREFERENCE_PAIRS if p["prompt"] != "YOUR PROMPT HERE"
]
print(f"Total pairs: {len(ALL_PAIRS)}  ({len(MY_PREFERENCE_PAIRS)} your own)")

## Part 2 — Train Your Reward Model

We train `distilbert-base-uncased` (66M params) as a reward model using TRL's `RewardTrainer`.

The **Bradley-Terry loss** pushes chosen scores above rejected scores:
$$\mathcal{L} = -\log \sigma\,(r_{\text{chosen}} - r_{\text{rejected}})$$

After training the model generalises: it can score *new* responses it has never seen.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer as RMTokenizer
from transformers import AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig

def build_reward_dataset(pairs):
    return Dataset.from_list([
        {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
        for p in pairs
    ])

reward_dataset = build_reward_dataset(ALL_PAIRS)
split = reward_dataset.train_test_split(test_size=0.2, seed=42)

# --- Reward-model tokenizer ------------------------------------------------
# DistilBERT is a BERT-family model: its special tokens are [CLS]/[SEP]/[PAD]
# and it has NO eos_token. Newer TRL appends an EOS to every chosen/rejected
# string during dataset prep, so a missing eos_token crashes with:
#     TypeError: endswith first arg must be str ... not NoneType
# Fix: give the tokenizer an eos_token (reuse [SEP]) and pass it explicitly
# to the trainer via processing_class.
reward_tokenizer = RMTokenizer.from_pretrained(REWARD_MODEL_ID)
if reward_tokenizer.eos_token is None:
    reward_tokenizer.eos_token = reward_tokenizer.sep_token
if reward_tokenizer.pad_token is None:
    reward_tokenizer.pad_token = reward_tokenizer.eos_token

# --- Reward model ----------------------------------------------------------
# Load the model explicitly with num_labels=1. The Bradley-Terry reward head
# must output a single scalar; passing a bare model-id string to RewardTrainer
# does not guarantee num_labels=1 across TRL versions, which would break the
# scalar scoring (logits[0, 0]) used in the cells below.
reward_base_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL_ID, num_labels=1
)
reward_base_model.config.pad_token_id = reward_tokenizer.pad_token_id

reward_config = RewardConfig(
    output_dir="./reward-model",
    num_train_epochs=15,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    max_length=256,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

reward_trainer = RewardTrainer(
    model=reward_base_model,
    args=reward_config,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=reward_tokenizer,
)

print(f"Training reward model on {len(split['train'])} pairs...")
reward_trainer.train()
reward_trainer.save_model("./reward-model")
print("Saved to ./reward-model/")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer as RMTokenizer

# Load trained reward model for scoring
reward_model = AutoModelForSequenceClassification.from_pretrained(
    "./reward-model", num_labels=1
).to(device).eval()
reward_tokenizer = RMTokenizer.from_pretrained(REWARD_MODEL_ID)


def score_response(prompt: str, response: str) -> float:
    """Scalar reward score — higher means your model prefers this response."""
    text = f"{prompt}\n\n{response}"
    inputs = reward_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=256
    ).to(device)
    with torch.no_grad():
        return reward_model(**inputs).logits[0, 0].item()


# Quick sanity check
print("Sanity check: does the reward model agree with your annotations?")
print()
correct, total = 0, 0
for pair in ALL_PAIRS:
    rc = score_response(pair["prompt"], pair["chosen"])
    rr = score_response(pair["prompt"], pair["rejected"])
    ok = rc > rr
    if ok:
        correct += 1
    total += 1
    icon = "OK" if ok else "X "
    print(f"  [{icon}] chosen={rc:+.2f}  rejected={rr:+.2f}  | '{pair['prompt'][:45]}...'")

print(f"\nAccuracy on own pairs: {correct}/{total} = {correct/total*100:.0f}%")
print("(Less than 100% is normal with a tiny dataset)")

## Part 3 — Explore Your Reward Function

### Fixed scoring: does the model rank responses intuitively?

First we score a set of hand-crafted responses ranging from bad to good.

In [ ]:
test_prompt = "How do I get better at public speaking?"

candidate_responses = [
    ("Wrong advice",
     "Avoid speaking in public until you feel 100% confident — then it will come naturally."),
    ("Too vague",
     "Public speaking is an important skill. Practice a lot and you will improve."),
    ("Decent",
     "Practice regularly and try to get feedback from others."),
    ("Good",
     "Join a group like Toastmasters for regular practice with feedback. Record yourself to spot habits you don't notice in the moment."),
    ("Detailed & actionable",
     "Great public speaking comes from deliberate practice. Join Toastmasters. Use the rule of three — three main points. Use intentional pauses (silence is power). Record yourself weekly and review critically. Breathe from the diaphragm. Make eye contact with different people, not a fixed point on the wall."),
]

labels, scores = [], []
for label, resp in candidate_responses:
    s = score_response(test_prompt, resp)
    labels.append(label)
    scores.append(s)

colors = ["green" if s == max(scores) else "salmon" if s == min(scores) else "steelblue" for s in scores]
plt.figure(figsize=(10, 4))
plt.barh(labels, scores, color=colors, alpha=0.85)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")
plt.xlabel("Reward score")
plt.title(f"Your Reward Model: '{test_prompt}'")
plt.tight_layout()
plt.show()

print("Green = highest, Red = lowest.")
print("Does the ranking match your intuition?")

### Interactive Reward Explorer

Now try it yourself. Type **any prompt** and **two responses** — your reward model will say which it prefers and by how much.

Try to find cases where it agrees and cases where it surprises you.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

style_wide = {"description_width": "90px"}
layout_full = widgets.Layout(width="100%")

prompt_box = widgets.Textarea(
    value="What is the best way to learn a new programming language?",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="65px"),
    style=style_wide,
)
response_a_box = widgets.Textarea(
    value="Just start coding and figure it out as you go.",
    description="Response A:",
    layout=widgets.Layout(width="100%", height="90px"),
    style=style_wide,
)
response_b_box = widgets.Textarea(
    value="Pick one small project and build it end-to-end. Use the official documentation. When stuck, look at Stack Overflow. Review your code after a week to see your own progress.",
    description="Response B:",
    layout=widgets.Layout(width="100%", height="90px"),
    style=style_wide,
)

score_out = widgets.Output()
score_btn = widgets.Button(
    description="Score Both",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="38px"),
)

def on_score(b):
    with score_out:
        score_out.clear_output(wait=True)
        sa = score_response(prompt_box.value, response_a_box.value)
        sb = score_response(prompt_box.value, response_b_box.value)
        winner = "A" if sa >= sb else "B"
        margin = abs(sa - sb)
        strength = "strongly" if margin > 0.4 else "slightly"
        print(f"  Response A: {sa:+.4f}")
        print(f"  Response B: {sb:+.4f}")
        print(f"  --> Your reward model {strength} prefers Response {winner} (margin {margin:.4f})")
        print()
        print("  Does this match your intuition?")
        print("  If not: what preference pair would teach the model to correct this?")

score_btn.on_click(on_score)

display(
    widgets.HTML("<hr><b>Interactive Reward Explorer</b> — type any prompt and two responses"),
    prompt_box,
    response_a_box,
    response_b_box,
    score_btn,
    score_out,
    widgets.HTML("<hr>"),
)

**Things to try in the explorer:**
- Does the model prefer detailed or concise responses? (varies by your annotations)
- Does it penalise factual errors?
- Does the prompt matter, or does it only look at the response?
- Try responses that are long but irrelevant — does it notice?
- Try switching A and B — does the score flip correctly?

## Part 4 — DPO Fine-Tuning of SmolLM2-135M

### DPO vs. PPO

| | PPO | DPO |
|---|---|---|
| Needs explicit reward model during LLM training | Yes | No |
| Separate RL training loop | Yes | No |
| Stability | Tricky | More stable |
| Memory | Higher | Lower |
| Used by | InstructGPT, early ChatGPT | Llama-2-Chat, Mistral-Instruct |

DPO directly optimises the policy to prefer chosen over rejected using a special loss — mathematically equivalent to PPO under a specific reward parameterisation but far simpler to implement.

We train on 500 examples from **UltraFeedback** — a 250K pair preference dataset rated by GPT-4.

In [ ]:
from datasets import load_dataset

print("Loading UltraFeedback (500 training, 50 eval examples)...")
dpo_train = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:500]")
dpo_eval  = load_dataset("trl-lib/ultrafeedback_binarized", split="test[:50]")

print(f"Train: {len(dpo_train)} | Eval: {len(dpo_eval)}")

# Show example structure
# NOTE: use single quotes for the dict keys inside these f-strings. Reusing the
# same quote char as the f-string delimiter (e.g. ["content"] inside f"...") is
# only valid on Python 3.12+ and raises SyntaxError on Colab's default 3.11.
ex = dpo_train[0]
print(f"\nExample:")
print(f"  prompt   : {str(ex['chosen'][0]['content'])[:80]}...")
print(f"  chosen   : {str(ex['chosen'][1]['content'])[:80]}...")
print(f"  rejected : {str(ex['rejected'][1]['content'])[:80]}...")

In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

dpo_config = DPOConfig(
    output_dir="./smollm2-dpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    report_to="none",
    warmup_ratio=0.1,
)

print("Starting DPO training (~15-25 min on T4)...")
print("Watch: rewards/margins should increase, loss should decrease.")
print()

dpo_trainer = DPOTrainer(
    model=MODEL_ID,
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    peft_config=peft_config,
)

n_trainable = sum(p.numel() for p in dpo_trainer.model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in dpo_trainer.model.parameters())
print(f"Trainable params: {n_trainable:,} / {n_total:,} ({n_trainable/n_total*100:.2f}%)")

dpo_trainer.train()
dpo_trainer.model.save_pretrained("./smollm2-dpo-adapter")
print("\nDPO training complete. Adapter saved.")

## Part 5 — Compare Before vs. After: Fixed Prompts

First the automated comparison on the three fixed prompts from Part 0.

In [ ]:
from peft import PeftModel

# Load both models for comparison
print("Loading models for comparison...")

base_cmp = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to(device).eval()

dpo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to(device)
dpo_model = PeftModel.from_pretrained(dpo_base, "./smollm2-dpo-adapter").eval()

print("Both models ready.")

In [ ]:
# Generate and score responses for the fixed comparison prompts
before_scores, after_scores = [], []
before_responses, after_responses = {}, {}

print("=" * 65)
print("Before vs. After DPO — Fixed Prompts")
print("=" * 65)

for prompt in COMPARISON_PROMPTS:
    r_before = generate(base_cmp, prompt, max_new_tokens=120)
    r_after  = generate(dpo_model, prompt, max_new_tokens=120)
    s_before = score_response(prompt, r_before)
    s_after  = score_response(prompt, r_after)

    before_responses[prompt] = r_before
    after_responses[prompt]  = r_after
    before_scores.append(s_before)
    after_scores.append(s_after)

    delta = s_after - s_before
    arrow = "improved" if delta > 0 else "declined"
    print(f"\nQ: {prompt}")
    print(f"  BEFORE ({s_before:+.3f}): {r_before[:180]}")
    print(f"  AFTER  ({s_after:+.3f}): {r_after[:180]}")
    print(f"  Reward {arrow} by {delta:+.3f}")

print()
print(f"Average reward BEFORE : {np.mean(before_scores):+.4f}")
print(f"Average reward AFTER  : {np.mean(after_scores):+.4f}")
print(f"Average improvement   : {np.mean(after_scores) - np.mean(before_scores):+.4f}")

In [ ]:
# Bar chart: before vs. after per prompt
x = np.arange(len(COMPARISON_PROMPTS))
short_labels = [p[:32] + "..." for p in COMPARISON_PROMPTS]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - 0.18, before_scores, 0.35, label="Before DPO", color="salmon",    alpha=0.85)
ax.bar(x + 0.18, after_scores,  0.35, label="After DPO",  color="steelblue", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=12, ha="right")
ax.set_ylabel("Reward score")
ax.set_title("Reward Scores: Before vs. After DPO Fine-Tuning")
ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
ax.legend()
plt.tight_layout()
plt.show()

### Interactive Before/After Comparison

Now try **your own prompts**. Type anything and see both models respond with live reward scores.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

cmp_prompt = widgets.Textarea(
    value="How do I stay motivated when learning something difficult?",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="65px"),
    style={"description_width": "70px"},
)
max_tok = widgets.IntSlider(
    value=120, min=40, max=250, step=10,
    description="Max tokens:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"),
)
temp = widgets.FloatSlider(
    value=0.7, min=0.1, max=1.5, step=0.1,
    description="Temperature:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"),
)

cmp_out = widgets.Output()
cmp_btn = widgets.Button(
    description="Generate & Compare",
    button_style="success",
    layout=widgets.Layout(width="200px", height="38px"),
)

def on_compare(b):
    with cmp_out:
        cmp_out.clear_output(wait=True)
        print("Generating... (10-20 sec)")
        p = cmp_prompt.value
        r_bef = generate(base_cmp,  p, max_new_tokens=max_tok.value, temperature=temp.value)
        r_aft = generate(dpo_model, p, max_new_tokens=max_tok.value, temperature=temp.value)
        s_bef = score_response(p, r_bef)
        s_aft = score_response(p, r_aft)
        delta = s_aft - s_bef
        arrow = "improved" if delta > 0 else "declined"

        print()
        print(f"Reward scores: Before={s_bef:+.3f} | After={s_aft:+.3f} | Delta={delta:+.3f} ({arrow})")
        print()
        print("="*55 + f"  BEFORE  (reward {s_bef:+.3f})")
        print(r_bef)
        print()
        print("="*55 + f"  AFTER   (reward {s_aft:+.3f})")
        print(r_aft)

cmp_btn.on_click(on_compare)

display(
    widgets.HTML("<hr><b>Interactive Before/After Comparison</b> — try any prompt"),
    cmp_prompt,
    widgets.HBox([max_tok, temp]),
    cmp_btn,
    cmp_out,
    widgets.HTML("<hr>"),
)

**Things to try:**
- Prompts very similar to your preference pairs — the model should improve most here
- Prompts outside your training distribution — did it generalise, or get worse?
- Sensitive topics (health, finance, relationships) — did safety improve?
- Increase temperature to 1.2+ — is the DPO model more or less stable under high randomness?
- The same prompt twice — is DPO model more consistent than the base model?

## Part 6 — Gradio Demo: Share with Others

`gradio` creates a web interface that runs inside Colab. With `share=True` you get a **public URL** you can open on any device or share with classmates — no installation required on their end.

The demo packages everything you built: reward scorer + before/after comparison + live reward scores.

In [ ]:
import gradio as gr

# ----------------------------------------------------------------
# Tab 1: Interactive reward scorer
# ----------------------------------------------------------------
def gradio_score(prompt, response_a, response_b):
    if not prompt.strip():
        return "Please enter a prompt.", "", ""
    sa = score_response(prompt, response_a)
    sb = score_response(prompt, response_b)
    winner = "A" if sa >= sb else "B"
    margin = abs(sa - sb)
    strength = "strongly" if margin > 0.4 else "slightly"
    verdict = f"Your reward model {strength} prefers Response {winner} (margin {margin:.3f})"
    return verdict, f"{sa:+.4f}", f"{sb:+.4f}"

scorer_tab = gr.Interface(
    fn=gradio_score,
    inputs=[
        gr.Textbox(label="Prompt",
                   value="How do I stop procrastinating?",
                   lines=2),
        gr.Textbox(label="Response A",
                   value="Just start. The hardest part is beginning.",
                   lines=4),
        gr.Textbox(label="Response B",
                   value="Break the task into 5-minute chunks. Commit to only 5 minutes. Most of the time you'll continue once you start — this is called the Zeigarnik effect.",
                   lines=4),
    ],
    outputs=[
        gr.Textbox(label="Verdict"),
        gr.Textbox(label="Score A"),
        gr.Textbox(label="Score B"),
    ],
    title="Your Reward Model",
    description="Score any two responses with the reward model you trained on YOUR preference pairs.",
)

# ----------------------------------------------------------------
# Tab 2: Before/after generator
# ----------------------------------------------------------------
def gradio_compare(prompt, max_tokens, temperature):
    if not prompt.strip():
        return "Please enter a prompt.", "", ""
    r_bef = generate(base_cmp,  prompt, max_new_tokens=int(max_tokens), temperature=float(temperature))
    r_aft = generate(dpo_model, prompt, max_new_tokens=int(max_tokens), temperature=float(temperature))
    s_bef = score_response(prompt, r_bef)
    s_aft = score_response(prompt, r_aft)
    delta = s_aft - s_bef
    summary = f"Base: {s_bef:+.3f}  |  DPO: {s_aft:+.3f}  |  Delta: {delta:+.3f}"
    return r_bef, r_aft, summary

compare_tab = gr.Interface(
    fn=gradio_compare,
    inputs=[
        gr.Textbox(label="Prompt",
                   value="How do I get better at public speaking?",
                   lines=3),
        gr.Slider(40, 250, value=120, step=10, label="Max new tokens"),
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature"),
    ],
    outputs=[
        gr.Textbox(label="Base model  (before DPO)", lines=7),
        gr.Textbox(label="Fine-tuned  (after DPO)",  lines=7),
        gr.Textbox(label="Reward scores"),
    ],
    title="SmolLM2-135M: Before vs. After DPO",
    description="Enter any prompt to compare base model vs. DPO fine-tuned model. Responses are scored by your reward model.",
)

# ----------------------------------------------------------------
# Tab 3: Reward model explainer — what makes a response score higher?
# ----------------------------------------------------------------
def gradio_probe(prompt, response, add_phrase):
    """Shows how appending a phrase changes the reward score."""
    original_score = score_response(prompt, response)
    modified_score = score_response(prompt, response + " " + add_phrase)
    delta = modified_score - original_score
    effect = "higher" if delta > 0 else "lower"
    return (
        f"{original_score:+.4f}",
        f"{modified_score:+.4f}",
        f"Adding the phrase made the score {effect} by {abs(delta):.4f}",
    )

probe_tab = gr.Interface(
    fn=gradio_probe,
    inputs=[
        gr.Textbox(label="Prompt",   value="How do I learn Python?", lines=2),
        gr.Textbox(label="Response", value="Read a book about it.", lines=4),
        gr.Textbox(label="Phrase to append",
                   value="I recommend starting with official Python.org tutorials and building a small project.",
                   lines=3),
    ],
    outputs=[
        gr.Textbox(label="Original score"),
        gr.Textbox(label="Score after appending"),
        gr.Textbox(label="Effect"),
    ],
    title="Probe Your Reward Model",
    description="See how adding a phrase changes the reward score. Great for understanding what words or patterns the model associates with high quality.",
)

# ----------------------------------------------------------------
# Launch
# ----------------------------------------------------------------
demo = gr.TabbedInterface(
    [scorer_tab, compare_tab, probe_tab],
    ["Reward Scorer", "Before vs After", "Probe Reward Model"],
)

demo.launch(share=True)   # share=True -> public URL, works from any device

The Gradio demo has three tabs:

| Tab | What it does |
|---|---|
| **Reward Scorer** | Score any two responses head-to-head with your reward model |
| **Before vs. After** | Generate + score responses from both models side by side |
| **Probe Reward Model** | See how appending a phrase changes the reward score — reveals what words/patterns the model associates with quality |

The public URL is valid for ~72 hours. You can open it on your phone, share with classmates, or present it.

## Challenge: Redesign Your Reward Function

The reward function encodes *someone's* values — not objective truth. Changing your preference pairs changes what the model optimises.

**Try one:**

**A) Conciseness experiment** — annotate pairs where the shorter answer is always chosen. Retrain. Does the reward model now penalise detailed responses? Use the Probe tab to check.

**B) Tone experiment** — chosen responses always use bullet points and numbered lists, rejected use prose. Does DPO change the output format?

**C) Safety experiment** — chosen responses always include a disclaimer ("consult a professional"), rejected give direct advice. Does the model become overly cautious on non-medical prompts?

In [ ]:
# Workflow for the challenge:
# 1. Edit MY_PREFERENCE_PAIRS above with your modified annotations
# 2. Re-run Part 2 (reward model training) — takes ~2 min
# 3. Use the Interactive Reward Explorer (Part 3) to see the new behaviour
#    without re-running DPO (which takes 20 min)
# 4. To see the full effect on the LLM, re-run Part 4 and 5.

# Shortcut: compare reward scores between original and new model
# on the same responses without retraining the LLM.

CHALLENGE_PROMPT = "What is the best way to improve your writing?"

CHALLENGE_RESPONSES = {
    "Concise bullet list":
        "1. Read widely. 2. Write daily — even one paragraph. 3. Get feedback. 4. Edit ruthlessly.",
    "Detailed prose":
        "Improving your writing is a long-term practice. The most effective approach combines wide reading across different genres and styles, consistent daily writing practice even when you feel you have nothing to say, actively seeking critical feedback from others, and developing a rigorous self-editing process. Each of these elements reinforces the others.",
    "Short but vague":
        "Just practice and you'll get better over time.",
}

print(f"Current reward scores for: '{CHALLENGE_PROMPT}'")
print()
for label, resp in CHALLENGE_RESPONSES.items():
    s = score_response(CHALLENGE_PROMPT, resp)
    print(f"  {label:<28}: {s:+.4f}")
print()
print("After retraining with your modified pairs, run this cell again.")
print("Did the ranking change in the direction you expected?")

## Summary

| What you built | Key insight |
|---|---|
| Preference dataset | Reward functions encode your values — not objective truth |
| Reward model (distilbert) | Bradley-Terry loss: chosen score must beat rejected score |
| Interactive reward explorer | The model generalises from your examples to unseen text |
| DPO fine-tuning (SmolLM2) | DPO = efficient RLHF without a separate RL training loop |
| Before/after comparison widget | Immediate feedback: does the model respond differently? |
| Gradio demo | Share and probe the model interactively from any device |

### The Bigger Picture

You reproduced in miniature what OpenAI, Anthropic, and Google do at scale:
- Their annotators label millions of pairs (your annotation exercise, at scale)
- Their reward model is trained on those labels (your distilbert, at scale)
- Their LLM is fine-tuned against the reward signal (your DPO, at scale)

**The open questions the field is actively working on:**
- What happens when the reward model is wrong or gameable?
- Whose preferences should we optimise for — and who decides?
- How do we prevent the model from finding reward hacks that score well but aren't actually good?

These questions start with the exact experiment you just ran.